# RRN GPU training — staged Vimeo-90K LMDB shards
Run each cell in order. The smoke test copies one shard to `/content` and runs 20 T4 steps. It never trains directly from Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lmdb


In [ ]:
from pathlib import Path
import json, subprocess
PROJECT = Path('/content/drive/MyDrive/train_ai/projects/RNN')
ADAPTER_REPO = 'https://github.com/koorock92/rrn-colab-adapter.git'
ADAPTER = Path('/content/rrn-colab-adapter')
UPSTREAM = Path('/content/RRN-upstream')
UPSTREAM_COMMIT = 'db440c5d363dfe6c11bdd6c82e63781b295e6211'
if not ADAPTER.exists():
    subprocess.run(['git', 'clone', '--depth', '1', ADAPTER_REPO, str(ADAPTER)], check=True)
if not UPSTREAM.exists():
    subprocess.run(['git', 'clone', 'https://github.com/junpan19/RRN.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', UPSTREAM_COMMIT], check=True)
WORK = ADAPTER / 'src'
SHARDS = Path('/content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_shards')
CACHE = Path('/content/rrn_shard_cache')
SMOKE_OUT = PROJECT / 'logs/sharded_gpu_smoke'
FULL_OUT = PROJECT / 'logs/sharded_gpu_full'
manifest = json.loads((SHARDS / 'manifest.json').read_text())
print('train shards:', len(manifest['train_shards']))
print('train samples:', manifest['train_samples'])
print('validation samples:', manifest['validation_samples'])


In [ ]:
import torch
assert torch.cuda.is_available(), '請先選 執行階段 > 變更執行階段類型 > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!df -h / /content/drive
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv


## 20-step real-model smoke test
This uses 128 channels and 10 blocks. `--no-prefetch` limits the first test to one training shard plus validation.

In [ ]:
import subprocess, sys
last = SMOKE_OUT / 'last.pt'
cmd = [sys.executable, str(WORK / 'train_rrn_sharded.py'),
       '--upstream-dir', str(UPSTREAM), '--shards-root', str(SHARDS), '--cache-dir', str(CACHE), '--output', str(SMOKE_OUT),
       '--device', 'cuda', '--scale', '4', '--crop-size', '64', '--batch-size', '4',
       '--workers', '2', '--channels', '128', '--blocks', '10', '--epochs', '70',
       '--save-every', '10', '--max-steps', '20', '--amp', '--no-prefetch']
if last.exists():
    cmd += ['--resume', str(last)]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
import json
lines = (SMOKE_OUT / 'metrics.jsonl').read_text().splitlines()
recent = [json.loads(line) for line in lines[-20:]]
print('steps:', len(recent))
print('mean seconds/step:', sum(x['seconds'] for x in recent) / len(recent))
print('last:', SMOKE_OUT / 'last.pt', (SMOKE_OUT / 'last.pt').exists())
print('best:', SMOKE_OUT / 'best.pt', (SMOKE_OUT / 'best.pt').exists())
!df -h / /content/drive
!nvidia-smi --query-gpu=name,memory.used,utilization.gpu --format=csv


## Full training (leave disabled until smoke-test speed is accepted)

In [ ]:
CONFIRM_FULL_TRAINING = False
if not CONFIRM_FULL_TRAINING:
    print('尚未啟動正式訓練。先確認上方 20-step 平均速度。')
else:
    import subprocess, sys
    last = FULL_OUT / 'last.pt'
    cmd = [sys.executable, str(WORK / 'train_rrn_sharded.py'),
           '--upstream-dir', str(UPSTREAM), '--shards-root', str(SHARDS), '--cache-dir', str(CACHE), '--output', str(FULL_OUT),
           '--device', 'cuda', '--scale', '4', '--crop-size', '64', '--batch-size', '4',
           '--workers', '2', '--channels', '128', '--blocks', '10', '--epochs', '70',
           '--save-every', '200', '--amp']
    if last.exists():
        cmd += ['--resume', str(last)]
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)
